In [ ]:

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

import statsmodels.api as sm
from statsmodels.stats.weightstats import DescrStatsW
from scipy import stats

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec


IR_PATH = "BDIR81FL.csv"
HR_PATH = "BDHR81FL.csv"
FIG_OUT = "BDHS2022_DBM_Figure.png"


print("=" * 70)
print("BDHS 2022 — Household Climate Vulnerability & Double Burden of")
print("Malnutrition Among Women of Reproductive Age in Bangladesh")
print("=" * 70)

print("\nLoading data ...")

IR_VARS = [
    "CASEID", "V001", "V002", "V003", "V005",
    "V012", "V024", "V025", "V106", "V190", "V445",
    "V743A", "V743B", "V743D", "V743F",
    "V169A", "V501", "V714", "V213",
]

HR_VARS = [
    "HV001", "HV002", "HV025",
    "HV201", "HV205", "HV206",
    "HV213", "HV214", "HV215",
    "HV009", "HV270", "SHDIST",
]

ir = pd.read_csv(IR_PATH, usecols=IR_VARS, low_memory=False)
hr = pd.read_csv(HR_PATH, usecols=HR_VARS, low_memory=False)
print(f"   IR: {ir.shape[0]:,} rows  |  HR: {hr.shape[0]:,} rows")


print("\nApplying inclusion criteria ...")

ir_bmi = ir[
    ir["V445"].notna() &
    ir["V445"].between(1000, 6000) &
    (ir["V213"] != 1)
].copy()
print(f"   Analytic sample: {len(ir_bmi):,} non-pregnant women with valid BMI")


print("\nMerging Individual and Household Recode files ...")

ir_bmi["HV001"] = ir_bmi["V001"]
ir_bmi["HV002"] = ir_bmi["V002"]
df = ir_bmi.merge(hr, on=["HV001", "HV002"], how="left")
print(f"   Merged sample: {len(df):,} women")


print("\nConstructing analysis variables ...")

df["wt"] = df["V005"] / 1_000_000

df["BMI"]              = df["V445"] / 100
df["underweight"]      = (df["BMI"] < 18.5).astype(int)
df["normal"]           = (df["BMI"].between(18.5, 24.999)).astype(int)
df["overweight"]       = (df["BMI"].between(25.0, 29.999)).astype(int)
df["obese"]            = (df["BMI"] >= 30.0).astype(int)
df["overweight_obese"] = (df["BMI"] >= 25.0).astype(int)

df["bmi_cat"] = pd.cut(
    df["BMI"],
    bins=[0, 18.5, 25.0, 30.0, 100],
    labels=["Underweight", "Normal", "Overweight", "Obese"],
    right=False
)

df["dep_water"] = (df["HV201"].notna() & (df["HV201"] >= 30)).astype(int)
df["dep_sanit"] = df["HV205"].isin({31, 42, 43, 96}).astype(int)
df["dep_floor"] = df["HV213"].isin({11, 12}).astype(int)
df["dep_wall"]  = df["HV214"].isin({11,12,13,21,22,23,24,25,26}).astype(int)
df["dep_roof"]  = df["HV215"].isin({11, 12, 13}).astype(int)
df["dep_elec"]  = (df["HV206"] == 0).astype(int)
df["dep_crowd"] = (df["HV009"] >= 6).astype(int)

CVI_COMPONENTS = ["dep_water", "dep_sanit", "dep_floor",
                  "dep_wall",  "dep_roof",  "dep_elec", "dep_crowd"]
df["CVI"] = df[CVI_COMPONENTS].sum(axis=1)

df["CVI_cat"] = pd.cut(
    df["CVI"], bins=[-1, 0, 1, 7],
    labels=["Low (CVI=0)", "Medium (CVI=1)", "High (CVI≥2)"]
)

DIVISION_MAP = {1:"Barishal", 2:"Chattogram", 3:"Dhaka",    4:"Khulna",
                5:"Mymensingh", 6:"Rajshahi", 7:"Rangpur",  8:"Sylhet"}
WEALTH_MAP   = {1:"Poorest", 2:"Poorer", 3:"Middle", 4:"Richer", 5:"Richest"}

df["division"] = df["V024"].map(DIVISION_MAP)
df["wealth"]   = df["V190"].map(WEALTH_MAP)
df["edu"]      = df["V106"].map({0:"None", 1:"Primary", 2:"Secondary", 3:"Higher"})
df["urban"]    = (df["HV025"] == 1).astype(int)
df["mobile"]   = (df["V169A"] == 1).astype(int)
df["age"]      = df["V012"]

df["age_grp"] = pd.cut(
    df["V012"], bins=[14, 24, 34, 44, 50],
    labels=["15–24", "25–34", "35–44", "45–49"]
)

AUTONOMY_VARS = ["V743A", "V743B", "V743D", "V743F"]
df["autonomy"] = df[AUTONOMY_VARS].apply(
    lambda row: sum(1 for v in AUTONOMY_VARS if pd.notna(row[v]) and row[v] in {1.0, 2.0}),
    axis=1
)

print(f"   Variables constructed. n = {len(df):,}  |  Weight sum = {df['wt'].sum():,.1f}")


N   = len(df)
wts = df["wt"].values
SEP = "─" * 70

def wpct(mask):
    w_n = wts[mask].sum()
    return int(mask.sum()), w_n / wts.sum() * 100

def wmean_sd(series):
    valid = series.notna()
    d = DescrStatsW(series[valid], weights=wts[valid], ddof=1)
    return d.mean, d.std

def weighted_chi2(col_cat, col_outcome):
    wct = pd.crosstab(col_cat, col_outcome, values=wts, aggfunc="sum").fillna(0)
    chi2, p, dof, _ = stats.chi2_contingency(wct)
    return chi2, p, dof


print(f"\n{'═'*70}")
print("TABLE 1 — WEIGHTED DESCRIPTIVE CHARACTERISTICS")
print(f"  n = {N:,}  |  Survey weight: V005 / 1,000,000")
print(f"{'═'*70}")

print("\n  Nutritional status (BMI, kg/m²)")
for lbl, mask in [
    ("Underweight (<18.5)",       df["underweight"] == 1),
    ("Normal (18.5–24.9)",        df["normal"] == 1),
    ("Overweight (25.0–29.9)",    df["overweight"] == 1),
    ("Obese (≥30.0)",             df["obese"] == 1),
    ("Overweight/obese combined", df["overweight_obese"] == 1),
]:
    n, pct = wpct(mask.values)
    print(f"    {lbl:<42}: {n:>5,}  ({pct:.1f}%)")

bmi_m, bmi_s = wmean_sd(df["BMI"])
print(f"\n  Mean BMI ± SD: {bmi_m:.2f} ± {bmi_s:.2f} kg/m²")

print("\n  Climate Vulnerability Index (CVI)")
for v in sorted(df["CVI"].dropna().unique()):
    n, pct = wpct((df["CVI"] == v).values)
    print(f"    CVI = {int(v)}: {n:>5,} ({pct:.1f}%)")
cvi_m, cvi_s = wmean_sd(df["CVI"])
print(f"  Mean CVI ± SD: {cvi_m:.2f} ± {cvi_s:.2f}")

print("\n  CVI category")
for lbl in ["Low (CVI=0)", "Medium (CVI=1)", "High (CVI≥2)"]:
    n, pct = wpct((df["CVI_cat"] == lbl).values)
    print(f"    {lbl:<25}: {n:>5,} ({pct:.1f}%)")

print("\n  CVI components (weighted prevalence deprived)")
for col, lbl in {
    "dep_water": "Unimproved water source",
    "dep_sanit": "Unimproved sanitation",
    "dep_floor": "Earthen/sand floor",
    "dep_wall":  "Vulnerable wall material",
    "dep_roof":  "Vulnerable roof material",
    "dep_elec":  "No electricity",
    "dep_crowd": "Overcrowding (≥6 members)",
}.items():
    n, pct = wpct((df[col] == 1).values)
    print(f"    {lbl:<35}: {n:>5,} ({pct:.1f}%)")

print("\n  Sociodemographic characteristics")
age_m, age_s = wmean_sd(df["age"])
print(f"    Mean age ± SD: {age_m:.1f} ± {age_s:.1f} years")
n_u, p_u = wpct((df["HV025"] == 1).values)
n_r, p_r = wpct((df["HV025"] == 2).values)
print(f"    Urban: {n_u:>5,} ({p_u:.1f}%)  |  Rural: {n_r:>5,} ({p_r:.1f}%)")

print("\n  Wealth quintile")
for k, v in WEALTH_MAP.items():
    n, pct = wpct((df["V190"] == k).values)
    print(f"    {v:<12}: {n:>5,} ({pct:.1f}%)")

print("\n  Education level")
for k, v in {0:"None", 1:"Primary", 2:"Secondary", 3:"Higher"}.items():
    n, pct = wpct((df["V106"] == k).values)
    print(f"    {v:<12}: {n:>5,} ({pct:.1f}%)")

print("\n  Women's agency")
n_m, p_m = wpct((df["mobile"] == 1).values)
print(f"    Mobile phone access: {n_m:>5,} ({p_m:.1f}%)")
aut_m, aut_s = wmean_sd(df["autonomy"])
print(f"    Mean autonomy score ± SD: {aut_m:.2f} ± {aut_s:.2f} (range 0–4)")


print(f"\n{'═'*70}")
print("TABLE 2 — WEIGHTED NUTRITIONAL STATUS BY CVI CATEGORY")
print(f"{'═'*70}")

header = f"{'CVI Category':<22} {'n':>6}  {'UW n (wtd%)':>14}  {'OW/OB n (wtd%)':>16}  {'Wtd Mean BMI':>12}"
print("\n  " + header)
print("  " + SEP)

ow_pcts, uw_pcts = [], []
for lbl in ["Low (CVI=0)", "Medium (CVI=1)", "High (CVI≥2)"]:
    mask   = (df["CVI_cat"] == lbl).values
    sub_wt = wts * mask
    n_cat  = mask.sum()
    uw_wp  = (df["underweight"].values * sub_wt).sum() / sub_wt.sum() * 100
    ow_wp  = (df["overweight_obese"].values * sub_wt).sum() / sub_wt.sum() * 100
    ow_pcts.append(ow_wp); uw_pcts.append(uw_wp)
    bmi_d  = DescrStatsW(df.loc[mask, "BMI"], weights=wts[mask], ddof=1)
    print(
        f"  {lbl:<22} {n_cat:>6,}  "
        f"{int((df['underweight'].values*mask).sum()):>4,} ({uw_wp:5.1f}%)  "
        f"{int((df['overweight_obese'].values*mask).sum()):>5,} ({ow_wp:5.1f}%)  "
        f"{bmi_d.mean:>12.2f}"
    )

chi2_uw, p_uw, dof_uw = weighted_chi2(df["CVI_cat"], df["underweight"])
chi2_ow, p_ow, dof_ow = weighted_chi2(df["CVI_cat"], df["overweight_obese"])
print(f"\n  Weighted χ² (underweight):      χ²={chi2_uw:.2f}, df={dof_uw}, {'p<0.001' if p_uw<0.001 else f'p={p_uw:.4f}'}")
print(f"  Weighted χ² (overweight/obese): χ²={chi2_ow:.2f}, df={dof_ow}, {'p<0.001' if p_ow<0.001 else f'p={p_ow:.4f}'}")


print(f"\n{'═'*70}")
print("TABLE 3 — SURVEY-WEIGHTED MULTIVARIABLE LOGISTIC REGRESSION")
print(f"{'═'*70}")

REG_VARS = ["CVI", "age", "V190", "V106", "urban", "mobile", "autonomy"]
REG_LABELS = {
    "CVI":      "CVI (continuous)",
    "age":      "Age (per year)",
    "V190":     "Wealth index (quintile)",
    "V106":     "Education level",
    "urban":    "Urban residence",
    "mobile":   "Mobile phone access",
    "autonomy": "Autonomy score (0–4)",
}

reg_df = df[["overweight_obese", "underweight", "wt"] + REG_VARS].dropna().copy()
print(f"\n  Regression sample: n = {len(reg_df):,}  |  Sum of weights: {reg_df['wt'].sum():,.1f}")

def run_weighted_logit(outcome_col, df_reg, label):
    X = sm.add_constant(df_reg[REG_VARS].astype(float))
    y = df_reg[outcome_col].astype(float)
    w = df_reg["wt"].astype(float)
    w_scaled = w / w.mean()
    model = sm.Logit(y, X, freq_weights=w_scaled).fit(disp=0, maxiter=200)
    params, ci, pvals = model.params, model.conf_int(), model.pvalues

    print(f"\n  Outcome: {label}")
    print(f"  {'Variable':<32} {'aOR':>7}  {'95% CI':>18}  {'p':>10}  Sig")
    print("  " + SEP)
    results = {}
    for var in REG_VARS:
        OR  = np.exp(params[var])
        lo  = np.exp(ci.loc[var, 0])
        hi  = np.exp(ci.loc[var, 1])
        pv  = pvals[var]
        sig = "***" if pv<0.001 else "**" if pv<0.01 else "*" if pv<0.05 else "ns"
        print(f"  {REG_LABELS[var]:<32} {OR:>7.3f}  ({lo:.3f}–{hi:.3f})  {pv:>10.4f}  {sig}")
        results[var] = {"OR": OR, "CI_lo": lo, "CI_hi": hi, "p": pv, "sig": sig}
    print(f"\n  Pseudo R² (McFadden): {model.prsquared:.4f}  |  AIC: {model.aic:.2f}")
    return results

res_ow = run_weighted_logit("overweight_obese", reg_df, "Overweight/Obese (ref = Normal weight)")
res_uw = run_weighted_logit("underweight",      reg_df, "Underweight (ref = Normal weight)")


print(f"\n{'═'*70}")
print("TABLE 4 — ERREYGERS CORRECTED CONCENTRATION INDEX (ECI)")
print(f"{'═'*70}")

def erreygers_ci(y, rank, weights=None):
    y    = np.asarray(y,    dtype=float)
    rank = np.asarray(rank, dtype=float)
    if weights is None:
        weights = np.ones_like(y)
    weights = np.asarray(weights, dtype=float)

    valid = ~(np.isnan(y) | np.isnan(rank) | np.isnan(weights))
    y, rank, weights = y[valid], rank[valid], weights[valid]
    w = weights / weights.sum()

    sort_idx  = np.argsort(rank, kind="stable")
    y_s, rank_s, w_s = y[sort_idx], rank[sort_idx], w[sort_idx]
    cum_w     = np.cumsum(w_s)
    frac_rank = cum_w - w_s / 2

    mu      = np.sum(w_s * y_s)
    cov_num = np.sum(w_s * y_s * frac_rank) - mu * np.sum(w_s * frac_rank)
    mean_r  = np.sum(w_s * frac_rank)
    var_r   = np.sum(w_s * (frac_rank - mean_r) ** 2)
    CI      = cov_num / var_r / mu * 2
    ECI     = 4 * mu * CI

    np.random.seed(42)
    n = len(y_s)
    ci_b, eci_b = np.empty(1000), np.empty(1000)
    for b in range(1000):
        idx       = np.random.choice(n, n, replace=True)
        yb, rb, wb = y_s[idx], rank_s[idx], w_s[idx]
        wb        = wb / wb.sum()
        cum_wb    = np.cumsum(wb)
        frb       = cum_wb - wb / 2
        mu_b      = np.sum(wb * yb)
        cov_b     = np.sum(wb * yb * frb) - mu_b * np.sum(wb * frb)
        var_b     = np.sum(wb * (frb - np.sum(wb * frb)) ** 2)
        if mu_b > 0 and var_b > 0:
            ci_b[b]  = cov_b / var_b / mu_b * 2
            eci_b[b] = 4 * mu_b * ci_b[b]
        else:
            ci_b[b] = eci_b[b] = np.nan

    se_CI  = np.nanstd(ci_b)
    se_ECI = np.nanstd(eci_b)
    p_ECI  = 2 * (1 - stats.norm.cdf(abs(ECI / se_ECI))) if se_ECI > 0 else np.nan

    return {
        "mu": mu, "CI": CI, "ECI": ECI,
        "se_ECI": se_ECI, "p_ECI": p_ECI,
        "CI_lo":  CI  - 1.96 * se_CI,
        "CI_hi":  CI  + 1.96 * se_CI,
        "ECI_lo": ECI - 1.96 * se_ECI,
        "ECI_hi": ECI + 1.96 * se_ECI,
    }

print("\n  Computing ECI with bootstrap standard errors (1,000 replications, seed=42) ...")

ci_ow = erreygers_ci(df["overweight_obese"], df["V190"], df["wt"])
ci_uw = erreygers_ci(df["underweight"],      df["V190"], df["wt"])

def sig_star(p):
    return "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns"

print(f"\n  {'Outcome':<20} {'Wtd mean':>8}  {'CI':>8}  {'CI (95% CI)':>22}  {'ECI':>8}  {'ECI (95% CI)':>22}  Sig")
print("  " + "─" * 100)
for lbl, res in [("Overweight/obese", ci_ow), ("Underweight", ci_uw)]:
    print(
        f"  {lbl:<20} {res['mu']:>8.3f}  "
        f"{res['CI']:>+8.4f}  ({res['CI_lo']:+.4f} to {res['CI_hi']:+.4f})  "
        f"{res['ECI']:>+8.4f}  ({res['ECI_lo']:+.4f} to {res['ECI_hi']:+.4f})  "
        f"{sig_star(res['p_ECI'])}"
    )
print("\n  ECI = 4 × weighted mean × CI  (Erreygers 2009). Ranked by DHS wealth quintile (V190).")
print("  ECI > 0: pro-rich concentration.  ECI < 0: pro-poor concentration.")


print(f"\n{'═'*70}")
print("TABLE 5 — WEIGHTED NUTRITIONAL STATUS AND CVI BY DIVISION")
print(f"{'═'*70}")

div_rows = []
for div in sorted(df["division"].dropna().unique()):
    sub = df[df["division"] == div]
    w_s = sub["wt"].values
    div_rows.append({
        "Division": div, "n": len(sub),
        "UW_%":      (sub["underweight"].values * w_s).sum() / w_s.sum() * 100,
        "OW_%":      (sub["overweight_obese"].values * w_s).sum() / w_s.sum() * 100,
        "Mean_BMI":  DescrStatsW(sub["BMI"],  weights=w_s, ddof=1).mean,
        "Mean_CVI":  DescrStatsW(sub["CVI"],  weights=w_s, ddof=1).mean,
    })

div_tab = pd.DataFrame(div_rows).sort_values("OW_%", ascending=False)
print(f"\n  {'Division':<14} {'n':>6}  {'UW % (wtd)':>12}  {'OW/OB % (wtd)':>14}  {'Mean BMI':>9}  {'Mean CVI':>9}")
print("  " + SEP)
for _, row in div_tab.iterrows():
    print(
        f"  {row['Division']:<14} {int(row['n']):>6,}  "
        f"{row['UW_%']:>11.1f}%  {row['OW_%']:>13.1f}%  "
        f"{row['Mean_BMI']:>9.2f}  {row['Mean_CVI']:>9.2f}"
    )


print(f"\n{'═'*70}")
print("TABLE 6 — WEIGHTED NUTRITIONAL STATUS BY WEALTH QUINTILE")
print(f"{'═'*70}")

print(f"\n  {'Wealth quintile':<12} {'n':>6}  {'UW % (wtd)':>12}  {'OW/OB % (wtd)':>15}  {'Mean CVI':>9}")
print("  " + "─" * 60)
for k in range(1, 6):
    sub = df[df["V190"] == k]
    w_s = sub["wt"].values
    print(
        f"  {WEALTH_MAP[k]:<12} {len(sub):>6,}  "
        f"{(sub['underweight'].values*w_s).sum()/w_s.sum()*100:>11.1f}%  "
        f"{(sub['overweight_obese'].values*w_s).sum()/w_s.sum()*100:>14.1f}%  "
        f"{DescrStatsW(sub['CVI'], weights=w_s, ddof=1).mean:>9.2f}"
    )


print(f"\n{'═'*70}")
print("TABLE 7 — SENSITIVITY ANALYSIS (CVI ≥ 1 subsample)")
print(f"{'═'*70}")

df_sens  = df[df["CVI"] >= 1].copy()
reg_sens = df_sens[["overweight_obese", "underweight", "wt"] + REG_VARS].dropna().copy()
print(f"\n  Restricted sample: n = {len(reg_sens):,}  (excluded CVI=0: n = {N - len(df_sens):,})")

run_weighted_logit("overweight_obese", reg_sens, "Overweight/Obese — CVI ≥ 1 subsample")
run_weighted_logit("underweight",      reg_sens, "Underweight — CVI ≥ 1 subsample")


print("\nGenerating Figure 1 ...")

RED    = "#C0392B"
BLUE   = "#2471A3"
GREY   = "#7F8C8D"
BLACK  = "#1a1a1a"
LRED   = "#F1948A"
LBLUE  = "#AED6F1"

fig = plt.figure(figsize=(20, 16), facecolor="white")
gs  = GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.38)

def style_ax(ax, title="", xlabel="", ylabel=""):
    ax.set_facecolor("white")
    ax.tick_params(colors=BLACK, labelsize=9)
    ax.xaxis.label.set_color(BLACK)
    ax.yaxis.label.set_color(BLACK)
    ax.title.set_color(BLACK)
    for sp in ax.spines.values():
        sp.set_edgecolor("#cccccc")
    ax.set_title(title, fontsize=10.5, fontweight="bold", pad=8, color=BLACK)
    if xlabel: ax.set_xlabel(xlabel, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, fontsize=9)
    ax.yaxis.grid(True, color="#eeeeee", linewidth=0.7)
    ax.set_axisbelow(True)

ax1 = fig.add_subplot(gs[0, 0])
wt_total = wts.sum()
bmi_vals = [
    (df["underweight"].values * wts).sum() / wt_total,
    (df["normal"].values * wts).sum() / wt_total,
    (df["overweight"].values * wts).sum() / wt_total,
    (df["obese"].values * wts).sum() / wt_total,
]
wedges, texts, autotexts = ax1.pie(
    bmi_vals,
    labels=["Underweight\n(<18.5)", "Normal\n(18.5–24.9)",
            "Overweight\n(25–29.9)", "Obese\n(≥30)"],
    autopct="%1.1f%%",
    colors=[BLUE, "#27AE60", "#F39C12", RED],
    startangle=90,
    pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor="white", linewidth=1.5)
)
for t in texts:     t.set_color(BLACK); t.set_fontsize(8)
for a in autotexts: a.set_color("white"); a.set_fontweight("bold"); a.set_fontsize(8)
ax1.set_title(f"(A) BMI distribution\n(n={N:,})", fontsize=10.5, fontweight="bold", color=BLACK)
ax1.set_facecolor("white")

ax2 = fig.add_subplot(gs[0, 1])
x = np.arange(3)
b1 = ax2.bar(x - 0.2, ow_pcts, width=0.35, color=RED,  alpha=0.85, edgecolor="none", label="Overweight/obese")
b2 = ax2.bar(x + 0.2, uw_pcts, width=0.35, color=BLUE, alpha=0.85, edgecolor="none", label="Underweight")
for b in list(b1) + list(b2):
    ax2.text(b.get_x() + b.get_width()/2, b.get_height() + 0.5,
             f"{b.get_height():.1f}%", ha="center", va="bottom",
             color=BLACK, fontsize=8, fontweight="bold")
ax2.set_xticks(x)
ax2.set_xticklabels(["Low\n(CVI=0)", "Medium\n(CVI=1)", "High\n(CVI≥2)"])
ax2.set_ylim(0, 60)
ax2.legend(fontsize=8, framealpha=0.9)
style_ax(ax2, "(B) Nutritional status by CVI category", "", "Weighted prevalence (%)")

ax3 = fig.add_subplot(gs[0, 2])
fvars = ["CVI", "age", "V190", "V106", "urban", "mobile", "autonomy"]
flbls = [REG_LABELS[v] for v in fvars]
or_ow = [res_ow[v]["OR"]    for v in fvars]
lo_ow = [res_ow[v]["CI_lo"] for v in fvars]
hi_ow = [res_ow[v]["CI_hi"] for v in fvars]
or_uw = [res_uw[v]["OR"]    for v in fvars]
lo_uw = [res_uw[v]["CI_lo"] for v in fvars]
hi_uw = [res_uw[v]["CI_hi"] for v in fvars]
y_pos = np.arange(len(fvars))
ax3.axvline(1.0, color="#aaaaaa", linewidth=1.2, linestyle="--")
for i in range(len(y_pos)):
    ax3.plot([lo_ow[i], hi_ow[i]], [i+0.15, i+0.15], color=RED,  lw=2.5)
    ax3.plot(or_ow[i],              i+0.15,            "D",        color=RED,  ms=7, zorder=5)
    ax3.plot([lo_uw[i], hi_uw[i]], [i-0.15, i-0.15], color=BLUE, lw=2.5)
    ax3.plot(or_uw[i],              i-0.15,            "s",        color=BLUE, ms=7, zorder=5)
ax3.set_yticks(y_pos)
ax3.set_yticklabels(flbls, fontsize=8.5)
ax3.legend(handles=[mpatches.Patch(color=RED, label="OW/OB"),
                    mpatches.Patch(color=BLUE, label="UW")],
           fontsize=8, loc="lower right")
style_ax(ax3, "(C) Forest plot — adjusted ORs", "aOR (95% CI)", "")

ax4 = fig.add_subplot(gs[1, 0])
wlbls = ["Poorest", "Poorer", "Middle", "Richer", "Richest"]
ow_w, uw_w = [], []
for k in range(1, 6):
    sub = df[df["V190"] == k]; ws = sub["wt"].values
    ow_w.append((sub["overweight_obese"].values * ws).sum() / ws.sum() * 100)
    uw_w.append((sub["underweight"].values * ws).sum() / ws.sum() * 100)
ax4.plot(wlbls, ow_w, "o-",  color=RED,  lw=2.5, ms=8, label="Overweight/obese")
ax4.plot(wlbls, uw_w, "s--", color=BLUE, lw=2.5, ms=8, label="Underweight")
ax4.fill_between(wlbls, ow_w, alpha=0.08, color=RED)
ax4.fill_between(wlbls, uw_w, alpha=0.08, color=BLUE)
ax4.set_ylim(0, 65)
ax4.set_xticklabels(wlbls, fontsize=8.5, rotation=15)
ax4.legend(fontsize=8)
style_ax(ax4, "(D) Nutritional status by wealth quintile", "", "Weighted prevalence (%)")

ax5 = fig.add_subplot(gs[1, 1])
def conc_curve(y, rank, weights):
    sort_idx = np.argsort(rank, kind="stable")
    y_s = np.asarray(y)[sort_idx]
    w_s = np.asarray(weights)[sort_idx]
    w_s = w_s / w_s.sum()
    return (np.concatenate([[0], np.cumsum(w_s)]),
            np.concatenate([[0], np.cumsum(w_s * y_s) / np.sum(w_s * y_s)]))

valid = df[["overweight_obese", "underweight", "V190", "wt"]].notna().all(axis=1)
df_v  = df[valid]
x_ow, y_ow = conc_curve(df_v["overweight_obese"], df_v["V190"], df_v["wt"])
x_uw, y_uw = conc_curve(df_v["underweight"],      df_v["V190"], df_v["wt"])
diag = np.linspace(0, 1, 100)
ax5.plot(diag, diag, "--", color=GREY, lw=1.5, label="Line of equality")
ax5.plot(x_ow, y_ow, color=RED,  lw=2.5, label=f"OW/OB (ECI={ci_ow['ECI']:+.3f})")
ax5.plot(x_uw, y_uw, color=BLUE, lw=2.5, label=f"UW   (ECI={ci_uw['ECI']:+.3f})")
ax5.fill_between(x_ow, np.interp(x_ow, diag, diag), y_ow, alpha=0.08, color=RED)
ax5.fill_between(x_uw, np.interp(x_uw, diag, diag), y_uw, alpha=0.08, color=BLUE)
ax5.set_xlim(0, 1); ax5.set_ylim(0, 1)
ax5.legend(fontsize=8)
style_ax(ax5, "(E) Concentration curves", "Cumulative wealth rank", "Cumulative outcome share")

ax6 = fig.add_subplot(gs[1, 2])
div_s  = div_tab.sort_values("OW_%")
dnames = div_s["Division"].tolist()
d_ow   = div_s["OW_%"].tolist()
colors = plt.cm.RdYlGn_r(np.array(d_ow) / max(d_ow))
bars   = ax6.barh(range(len(dnames)), d_ow, color=colors, alpha=0.85, edgecolor="none")
ax6.set_yticks(range(len(dnames)))
ax6.set_yticklabels(dnames, fontsize=8.5)
for b in bars:
    ax6.text(b.get_width() + 0.3, b.get_y() + b.get_height()/2,
             f"{b.get_width():.1f}%", va="center", fontsize=8)
style_ax(ax6, "(F) OW/OB prevalence by division", "Weighted prevalence (%)", "")

ax7 = fig.add_subplot(gs[2, 0])
cnames = ["Unimp.\nWater", "Unimp.\nSanit.", "Earth\nFloor",
          "Vuln.\nWalls", "Vuln.\nRoof", "No\nElec.", "Overcrowding\n(≥6)"]
cpcts  = [(df[c].values * wts).sum() / wts.sum() * 100 for c in CVI_COMPONENTS]
cclrs  = [RED if p > 10 else "#E67E22" if p > 3 else BLUE for p in cpcts]
bars7  = ax7.bar(range(len(cnames)), cpcts, color=cclrs, alpha=0.85, edgecolor="none")
ax7.set_xticks(range(len(cnames)))
ax7.set_xticklabels(cnames, fontsize=8, rotation=10)
for b in bars7:
    ax7.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3,
             f"{b.get_height():.1f}%", ha="center", va="bottom", fontsize=7.5)
style_ax(ax7, "(G) CVI components — weighted prevalence", "", "Weighted prevalence (%)")

ax8 = fig.add_subplot(gs[2, 1])
auto_ow, auto_uw = [], []
for s in range(5):
    sub = df[df["autonomy"] == s]; ws = sub["wt"].values
    auto_ow.append((sub["overweight_obese"].values * ws).sum() / ws.sum() * 100 if len(sub) > 20 else np.nan)
    auto_uw.append((sub["underweight"].values * ws).sum() / ws.sum() * 100       if len(sub) > 20 else np.nan)
ax8.plot(range(5), auto_ow, "o-",  color=RED,  lw=2.5, ms=8, label="Overweight/obese")
ax8.plot(range(5), auto_uw, "s--", color=BLUE, lw=2.5, ms=8, label="Underweight")
ax8.set_xticks(range(5))
ax8.set_xticklabels(["0\n(None)", "1", "2", "3", "4\n(Full)"], fontsize=8.5)
ax8.legend(fontsize=8)
style_ax(ax8, "(H) Nutritional status by autonomy score", "Women's autonomy score", "Weighted prevalence (%)")

ax9 = fig.add_subplot(gs[2, 2])
eci_vals = [ci_ow["ECI"], ci_uw["ECI"]]
eci_errs = [[ci_ow["ECI"] - ci_ow["ECI_lo"], ci_uw["ECI"] - ci_uw["ECI_lo"]],
            [ci_ow["ECI_hi"] - ci_ow["ECI"], ci_uw["ECI_hi"] - ci_uw["ECI"]]]
bars9 = ax9.bar(["OW/OB", "Underweight"], eci_vals,
                color=[RED, BLUE], alpha=0.85, edgecolor="none", width=0.45)
ax9.errorbar(["OW/OB", "Underweight"], eci_vals,
             yerr=eci_errs, fmt="none", color=BLACK, capsize=6, lw=2)
ax9.axhline(0, color=GREY, lw=1.2, linestyle="--")
for bar, val, p in zip(bars9, eci_vals, [ci_ow["p_ECI"], ci_uw["p_ECI"]]):
    sig = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns"
    ax9.text(bar.get_x() + bar.get_width()/2,
             val + (0.02 if val >= 0 else -0.03),
             f"{val:+.3f} ({sig})",
             ha="center", va="bottom" if val >= 0 else "top",
             fontsize=9, fontweight="bold")
ax9.text(0.5, 0.05, "ECI > 0: pro-rich  |  ECI < 0: pro-poor",
         transform=ax9.transAxes, ha="center", va="bottom", fontsize=7.5, color=GREY)
style_ax(ax9, "(I) Erreygers Corrected Concentration Index", "", "ECI value")

fig.suptitle(
    "Figure 1. Survey-weighted analysis of the double burden of malnutrition and climate\n"
    "vulnerability among women of reproductive age, BDHS 2022 (n=9,348)",
    fontsize=12, fontweight="bold", color=BLACK, y=0.995
)

plt.savefig(FIG_OUT, dpi=300, bbox_inches="tight", facecolor="white")
print(f"   Figure saved → {FIG_OUT}")


print(f"\n{'═'*70}")
print("Summary of Key Findings")
print(f"{'═'*70}")
print(f"""
  Sample: {N:,} non-pregnant women with valid BMI (V213≠1, V445: 1,000–6,000)
  Survey weight: V005 / 1,000,000

  Nutritional status (weighted)
    Overweight/obese : {(df['overweight_obese'].values*wts).sum()/wts.sum()*100:.1f}%
    Underweight      : {(df['underweight'].values*wts).sum()/wts.sum()*100:.1f}%

  CVI gradient (weighted)
    OW/OB : {ow_pcts[0]:.1f}% (CVI=0)  →  {ow_pcts[1]:.1f}% (CVI=1)  →  {ow_pcts[2]:.1f}% (CVI≥2)
    UW    : {uw_pcts[0]:.1f}% (CVI=0)  →  {uw_pcts[1]:.1f}% (CVI=1)  →  {uw_pcts[2]:.1f}% (CVI≥2)

  Multivariable logistic regression (key exposure: CVI)
    OW/OB : aOR={res_ow['CVI']['OR']:.3f} (95%CI {res_ow['CVI']['CI_lo']:.3f}–{res_ow['CVI']['CI_hi']:.3f}), p={res_ow['CVI']['p']:.4f} {res_ow['CVI']['sig']}
    UW    : aOR={res_uw['CVI']['OR']:.3f} (95%CI {res_uw['CVI']['CI_lo']:.3f}–{res_uw['CVI']['CI_hi']:.3f}), p={res_uw['CVI']['p']:.4f} {res_uw['CVI']['sig']}

  Erreygers Corrected Concentration Index
    OW/OB : ECI={ci_ow['ECI']:+.4f} (95%CI {ci_ow['ECI_lo']:+.4f}–{ci_ow['ECI_hi']:+.4f}), {sig_star(ci_ow['p_ECI'])}
    UW    : ECI={ci_uw['ECI']:+.4f} (95%CI {ci_uw['ECI_lo']:+.4f}–{ci_uw['ECI_hi']:+.4f}), {sig_star(ci_uw['p_ECI'])}

""")